# 🌊 RAINWISE V3.1: SegFormer High-Performance Sprint
This notebook is optimized for training the SegFormer-B0 hierarchical transformer on Google Colab GPUs (T4/L4/A100).

### 🛠️ Step 1: Environment Setup

In [ ]:
!pip install timm tqdm opencv-python segmentation-models-pytorch
import torch
print(f"📡 Active GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE'}")

from google.colab import drive
drive.mount('/content/drive')

### 📦 Step 2: Extract Project Data
Ensure you have uploaded `rainwise_v3_1.zip` to your Google Drive root.

In [ ]:
import os
!rm -rf /content/project

zip_path = "/content/drive/MyDrive/rainwise_v3_1.zip"
if os.path.exists(zip_path):
    print("📦 Extracting dataset...")
    !unzip -q {zip_path} -d /content/
    %cd /content/project/src
else:
    print("❌ ERROR: rainwise_v3_1.zip not found in Drive root!")

### 💉 Step 3: Inject Latest Source Code
Instead of re-zipping, we are overwriting `train_custom.py` and `losses.py` with the latest optimized versions directly.

In [ ]:
# 1. Inject Updated losses.py (AMP-Safe)
loss_content = """import torch
import torch.nn as nn
import torch.nn.functional as F

class DiceFocalLoss(nn.Module):
    def __init__(self, alpha=0.25, gamma=2.0, smooth=1.0):
        super(DiceFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.smooth = smooth

    def dice_loss(self, inputs, targets, smooth=1):
        inputs = torch.sigmoid(inputs)
        inputs = inputs.reshape(-1)
        targets = targets.reshape(-1)
        intersection = (inputs * targets).sum()
        return 1 - (2. * intersection + smooth) / (inputs.sum() + targets.sum() + smooth)

    def forward(self, pred, target):
        num_classes = pred.shape[1]
        total_loss = 0
        pred = pred.float() # Stability
        
        for i in range(num_classes):
            logits = pred[:, i]
            p = torch.sigmoid(logits)
            t = (target == i).float()
            
            # Use with_logits for stability in AMP
            bce = F.binary_cross_entropy_with_logits(logits, t, reduction='none')
            p_t = p * t + (1 - p) * (1 - t)
            f_loss = self.alpha * (1 - p_t)**self.gamma * bce
            f_loss = f_loss.mean()
            
            d_loss = self.dice_loss(logits, t, self.smooth)
            total_loss += (0.5 * f_loss + 0.5 * d_loss)
            
        return total_loss\"""

with open('/content/project/src/training/losses.py', 'w') as f:
    f.write(loss_content)

# 2. Inject Updated train_custom.py
script_content = """import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from tqdm import tqdm
import numpy as np
from pathlib import Path
import os
import sys
import cv2

sys.path.append(str(Path(__file__).parent.parent))

from models.segmentation_v2 import create_deeplabv3plus as FloodNet
from models.flood_transformer import SwinFloodNet
from models.segformer_model import SegFormerFlood
from models.custom_dataset import FloodCustomDataset
from training.losses import DiceFocalLoss

def calculate_metrics(pred, target, threshold=0.3):
    pred = torch.sigmoid(pred)
    pred = (pred > threshold).float()
    p = pred[:, 1].cpu().numpy().flatten()
    t = (target == 1).cpu().numpy().flatten()
    intersection = np.logical_and(p, t).sum()
    union = np.logical_or(p, t).sum()
    iou = intersection / (union + 1e-6)
    dice = (2. * intersection) / (p.sum() + t.sum() + 1e-6)
    precision = intersection / (p.sum() + 1e-6)
    recall = intersection / (t.sum() + 1e-6)
    return iou, dice, precision, recall

def save_visual_check(epoch, data, mask, output, save_dir, threshold=0.3):
    save_dir = Path(save_dir); save_dir.mkdir(parents=True, exist_ok=True)
    img = (data[0, :3].permute(1, 2, 0).cpu().numpy() * 255).astype(np.uint8)
    img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
    gt = cv2.cvtColor((mask[0].cpu().numpy() * 255).astype(np.uint8), cv2.COLOR_GRAY2BGR)
    pred = (torch.sigmoid(output[0, 1]).cpu().numpy() > threshold).astype(np.uint8) * 255
    pred = cv2.cvtColor(pred, cv2.COLOR_GRAY2BGR)
    gt[mask[0].cpu().numpy() == 1] = [0, 255, 0]
    pred[pred[:,:,0] == 255] = [0, 0, 255]
    cv2.imwrite(str(save_dir / f\"epoch_{epoch+1}_v3_1.png\"), np.hstack([img, gt, pred]))

def start_training():
    device = torch.device(\"cuda\")
    PROJECT_ROOT = Path(__file__).parent.parent.parent
    DATASET_ROOT = PROJECT_ROOT / \"dataset_split\"
    WEIGHTS_DIR = PROJECT_ROOT / \"weights\"; WEIGHTS_DIR.mkdir(exist_ok=True)
    VISUAL_DIR = PROJECT_ROOT / \"visual_val_v3_1\"
    
    NUM_CLASSES = 2
    BATCH_SIZE = int(os.getenv(\"BATCH_SIZE\", 8))
    LEARNING_RATE = 1e-4
    EPOCHS = int(os.getenv(\"EPOCHS\", 40))
    USE_AMP = os.getenv(\"USE_AMP\", \"0\") == \"1\"
    
    train_dataset = FloodCustomDataset(str(DATASET_ROOT / \"train\" / \"images\"), 
                                       masks_dir=str(DATASET_ROOT / \"train\" / \"masks_hq\"), balance=True)
    val_dataset = FloodCustomDataset(str(DATASET_ROOT / \"val\" / \"images\"), 
                                     masks_dir=str(DATASET_ROOT / \"val\" / \"masks_hq\"), balance=False)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, persistent_workers=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, persistent_workers=True)

    model_type = os.getenv(\"MODEL_TYPE\", \"segformer\")
    if model_type == \"segformer\":
        model = SegFormerFlood(num_classes=NUM_CLASSES, in_channels=6).to(device)
        weights_name = \"segformer_flood_v3_1.pth\"
    
    optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=3)
    criterion = DiceFocalLoss()
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    
    best_iou = 0.0
    for epoch in range(EPOCHS):
        model.train(); train_loss = 0
        pbar = tqdm(train_loader, desc=f\"Epoch {epoch+1}/{EPOCHS}\")
        for data, mask, _ in pbar:
            data, mask = data.to(device), mask.to(device)
            optimizer.zero_grad()
            with torch.cuda.amp.autocast(enabled=USE_AMP):
                output = model(data)
                loss = criterion(output, mask)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            train_loss += loss.item()
            pbar.set_postfix({\"loss\": f\"{loss.item():.4f}\"})

        model.eval(); v_metrics = {\"loss\": 0, \"iou\": 0, \"dice\": 0, \"prec\": 0, \"rec\": 0}
        with torch.no_grad():
            for i, (data, mask, _) in enumerate(val_loader):
                data, mask = data.to(device), mask.to(device)
                output = model(data); v_metrics[\"loss\"] += criterion(output, mask).item()
                iou, dice, prec, rec = calculate_metrics(output, mask)
                v_metrics[\"iou\"] += iou; v_metrics[\"dice\"] += dice; v_metrics[\"prec\"] += prec; v_metrics[\"rec\"] += rec
                if i == 0: save_visual_check(epoch, data, mask, output, VISUAL_DIR)
        num_v = len(val_loader); avg_iou = v_metrics['iou']/num_v
        print(f\"📊 Epoch {epoch+1} Summary: IoU: {avg_iou:.4f}\")
        scheduler.step(avg_iou)
        if avg_iou > best_iou:
            best_iou = avg_iou
            torch.save(model.state_dict(), str(WEIGHTS_DIR / weights_name))
if __name__ == \"__main__\":
    start_training()\"""

with open('/content/project/src/training/train_custom.py', 'w') as f:
    f.write(script_content)
print("💉 Injection Complete. All scripts updated for AMP stability.")

### 🚀 Step 4: Launch V3.1 Training
We are using **Mixed Precision (AMP)** and a larger **Batch Size (32)** for maximum speed.

In [ ]:
os.environ['MODEL_TYPE'] = 'segformer'
os.environ['BATCH_SIZE'] = '32'  # Optimized for T4 GPU
os.environ['USE_AMP'] = '1'     # Enable Mixed Precision

!python training/train_custom.py

### 🖼️ Step 5: Visual Validation
Check the latest visual output from the training run.

In [ ]:
import matplotlib.pyplot as plt
import cv2
from glob import glob

visuals = sorted(glob("/content/project/visual_val_v3_1/*.png"))
if visuals:
    img = cv2.imread(visuals[-1])
    plt.figure(figsize=(15, 5))
    plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    plt.title(f"Latest Validation: {os.path.basename(visuals[-1])}")
    plt.axis('off')
    plt.show()
else:
    print("Waiting for first epoch visuals...")

### 📥 Step 6: Save Final Weights
Once training finishes, copy the weights back to your Drive.

In [ ]:
!cp /content/project/weights/segformer_flood_v3_1.pth /content/drive/MyDrive/RAINWISE_V3_1_FINAL.pth
print("✅ Training Complete. Weights saved to Drive.")